# Step 6 (Kaggle / GPU) — build batch_XXX.npz unattended

Runs on a **free Kaggle GPU**, server-side, so you can **Save & Run All, close your
laptop**, and collect the `.npz` files later. It is **resumable**: batches already
saved are skipped, so a cutoff never wastes work.

## One-time setup on Kaggle
1. **Upload your videos as a Dataset**: zip `Dataset/videos/` and create a new
   Kaggle *Dataset* from the zip. (~2 GB; you only need the videos, not the 30 GB
   of author features.)
2. **New Notebook** → upload this `.ipynb` (File → Import Notebook).
3. **Enable GPU**: right panel → *Settings* → *Accelerator* → **GPU T4 x1**.
4. **Attach your dataset**: right panel → *Input* → *Add Input* → your videos dataset.
5. Set `VIDEOS_ROOT` in the config cell to where your videos landed
   (shown under */kaggle/input/...* in the Input panel).
6. **Save Version → Save & Run All (Commit)** — it now runs in the background.
7. When done, download results from the committed version's *Output* tab
   (`/kaggle/working/features_generated/...`).

## 1. GPU check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Enable it: Settings -> Accelerator -> GPU T4 x1')

## 2. Config - paths and settings (set once)
Set `VIDEOS_ROOT` / `OUT_ROOT` once here. The split, number of batches and frames are passed to `build_features()` later, so you don't edit this cell per run.

In [ ]:
import glob, os

# ----- environment: set these ONCE -----
VIDEOS_ROOT = '/kaggle/input/datasets/njsolanki/dashcam-videos/Kaggle_Dataset'  # EDIT to your dataset
OUT_ROOT    = '/kaggle/working/features_generated'
ACC_MIN, ACC_MAX = 3, 5      # accidents per batch (the rest are non-accident)
SEED = 1234                   # fixed seed -> reproducible AND resumable
# ----------------------------------------

# quick path check (both splits)
for sp in ['train', 'test']:
    na = len(glob.glob(f'{VIDEOS_ROOT}/{sp}/Accident/*.mp4'))
    nn = len(glob.glob(f'{VIDEOS_ROOT}/{sp}/NonAccident/*.mp4'))
    print(f'{sp}: {na} accident, {nn} non-accident videos')
if len(glob.glob(f'{VIDEOS_ROOT}/*/*/*.mp4')) == 0:
    print('\n!! No videos found. Fix VIDEOS_ROOT. Under /kaggle/input:')
    for p in glob.glob('/kaggle/input/*'):
        print('   ', p)

## 3. Models + helpers (GPU) + per-video encoder

In [ ]:
import cv2, time
import numpy as np
import torch.nn as nn

N_FRAMES=100; N_DETECTION=20; N_OBJECTS=19; FEAT_DIM=4096
KEEP={1,2,3,4,6,7,8}; SCORE_THRESH=0.5
MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)

from torchvision.models.detection import fasterrcnn_resnet50_fpn
try:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
except Exception:
    detector = fasterrcnn_resnet50_fpn(pretrained=True)
detector.eval().to(device)

from torchvision.models import vgg16
try:
    from torchvision.models import VGG16_Weights
    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
except Exception:
    vgg = vgg16(pretrained=True)
vgg.eval()
vgg_features=vgg.features.to(device); vgg_avgpool=vgg.avgpool.to(device)
vgg_fc7=nn.Sequential(*list(vgg.classifier.children())[:5]).to(device)

def detect_boxes(frame):
    t=torch.from_numpy(frame).permute(2,0,1).float().to(device)/255.0
    with torch.no_grad():
        out=detector([t])[0]
    b=out['boxes'].cpu().numpy(); l=out['labels'].cpu().numpy(); s=out['scores'].cpu().numpy()
    kept=[(float(sc),bx) for bx,lb,sc in zip(b,l,s) if sc>=SCORE_THRESH and int(lb) in KEEP]
    kept.sort(key=lambda z:z[0], reverse=True)
    return [bx for _,bx in kept[:N_OBJECTS]]

def crop_box(frame, box):
    h,w,_=frame.shape; x1,y1,x2,y2=box
    x1=max(0,int(round(x1))); y1=max(0,int(round(y1)))
    x2=min(w,int(round(x2))); y2=min(h,int(round(y2)))
    return None if (x2-x1<2 or y2-y1<2) else frame[y1:y2,x1:x2]

def to_features_batch(crops):
    if not crops: return np.zeros((0,FEAT_DIM),np.float32)
    batch=np.stack([cv2.resize(c,(224,224)) for c in crops]).astype(np.float32)/255.0
    x=torch.from_numpy(batch).permute(0,3,1,2).to(device)
    x=(x-MEAN)/STD
    with torch.no_grad():
        x=vgg_features(x); x=vgg_avgpool(x); x=torch.flatten(x,1); x=vgg_fc7(x)
    return x.cpu().numpy().astype(np.float32)

def encode_video(path, max_frames=100):
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ok,f=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    cap.release()
    out=np.zeros((N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    for t in range(min(max_frames,len(frames))):
        frame=frames[t]; crops=[frame]
        for box in detect_boxes(frame):
            c=crop_box(frame,box)
            if c is not None: crops.append(c)
            if len(crops)==N_DETECTION: break
        feats=to_features_batch(crops)
        out[t,:feats.shape[0],:]=feats
    return out
print('models ready on', device)

## 4. Define `build_features(split, num_batches, max_frames)`
One call builds a whole split. Resumable (already-saved batches are skipped) and deterministic per batch index, so a resumed commit reproduces the same batches.

In [ ]:
import random

def build_features(split, num_batches, max_frames, batch_from=1):
    """Build `num_batches` .npz files for `split`, each video encoded with `max_frames` frames.
    Each batch = random ACC_MIN..ACC_MAX accidents + rest non-accident, random order.
    Resumable: already-saved batches are skipped. Example: build_features('test', 10, 100)."""
    acc_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{split}/Accident/*.mp4'))
    non_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{split}/NonAccident/*.mp4'))
    assert acc_pool and non_pool, f'no videos for split={split} - check VIDEOS_ROOT'

    def _n_acc(b):
        return random.Random(SEED + b).randint(ACC_MIN, ACC_MAX)

    def plan_batch(b):            # 0-based; deterministic -> resumable
        rng   = random.Random(SEED + b)
        n_acc = rng.randint(ACC_MIN, ACC_MAX)
        n_non = 10 - n_acc
        acc_off = sum(_n_acc(j) for j in range(b))
        non_off = sum(10 - _n_acc(j) for j in range(b))
        acc = [acc_pool[(acc_off + k) % len(acc_pool)] for k in range(n_acc)]
        non = [non_pool[(non_off + k) % len(non_pool)] for k in range(n_non)]
        items = [(p, (0.0, 1.0)) for p in acc] + [(p, (1.0, 0.0)) for p in non]
        rng.shuffle(items)
        return items

    out_dir = f'{OUT_ROOT}/{split}'
    os.makedirs(out_dir, exist_ok=True)
    for b in range(batch_from, num_batches + 1):
        out_path = f'{out_dir}/batch_{b:03d}.npz'
        if os.path.exists(out_path):
            print(f'skip batch_{b:03d} (already done)'); continue
        items  = plan_batch(b - 1)
        data   = np.zeros((10, N_FRAMES, N_DETECTION, FEAT_DIM), np.float32)
        labels = np.zeros((10, 2), np.float32)
        t0 = time.time(); print(f'=== {split} batch_{b:03d} ===')
        for i, (p, lab) in enumerate(items):
            ti = time.time(); data[i] = encode_video(p, max_frames); labels[i] = lab
            print(f'  [{i+1:2d}/10] {"A" if lab[1]==1 else "n"} {os.path.basename(p):12s} {time.time()-ti:5.1f}s')
        np.savez(out_path, data=data.astype(np.float16), labels=labels.astype(np.float16))
        print(f'  saved {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)  total {time.time()-t0:.1f}s\n')
    print(f'DONE: {split} ({num_batches} batches)')
    return out_dir

## 5. Run
Call it with the split, number of batches and frames you want, e.g. `build_features('test', 10, 100)`. For unattended runs: set the call, then **Save Version -> Save & Run All**.

In [ ]:
# build one split in one call:  build_features(split, num_batches, max_frames)
build_features('test', 10, 100)

# examples:
#   build_features('train', 20, 100)
#   build_features('test', 10, 100, batch_from=5)   # resume from batch 5

## 6. Verify a saved batch

In [ ]:
def verify(split, b=1):
    f = np.load(f'{OUT_ROOT}/{split}/batch_{b:03d}.npz')
    labels = f['labels']; n_acc = int(sum(r[1] == 1 for r in labels))
    print('data  :', f['data'].shape, f['data'].dtype)
    print('labels:', labels.shape)
    print('composition:', n_acc, 'accident +', 10 - n_acc, 'non-accident')
    print('order :', ['A' if r[1] == 1 else 'n' for r in labels])
    print('data >= 0 ?', bool((f['data'] >= 0).all()))

verify('test', 1)

## Running the whole dataset unattended
- Edit the run cell, e.g. `build_features('train', 20, 100)` or `build_features('test', 10, 100)`.
- **Save Version -> Save & Run All (Commit)** and close your laptop.
- If a commit runs out of time, call it again with `batch_from=` the next batch (finished batches are skipped anyway).
- Download outputs from the version's **Output** tab (`/kaggle/working/features_generated/...`).